The user asks a question. Relevant information is retrieved from the vector database and passed to the LLM as context, which then generates the answer.

In [1]:
# pip install langchain-groq python-dotenv

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.load_local(
    "../data/faiss_index", embedding_model,
    allow_dangerous_deserialization=True
)
print("Loaded chunks:", vector_store.index.ntotal)


c:\Users\sahil\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\sahil\AppData\Local\Temp\ipykernel_30596\1720079251.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
W1121 18:02:22.184000 30596 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded chunks: 680


In [3]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv("../.env")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)


In [4]:
history = []

def rewrite_question(question):
    if not history:
        return question
    history_text = "\n".join(
        f"User: {q}\nAssistant: {a}" for q, a in history[-3:]
    )
    rw_prompt = f"""Given this conversation history:
{history_text}

Rewrite the follow-up question as a complete standalone question that can be understood without the history. Keep the same meaning. Write only the rewritten question.

Follow-up question: {question}"""
    return llm.invoke(rw_prompt).content.strip()


In [5]:
THRESHOLD = 0.45

def ask(question):
    standalone = rewrite_question(question)
    if standalone != question:
        print("The Understood question:", standalone)

    results = vector_store.similarity_search_with_relevance_scores(standalone, k=3)
    best_score = results[0][1]
    print("Best score:", round(best_score, 2))

    if best_score < THRESHOLD:
        print("Answer: I don't have this information in the documents.")
        print("Action: Escalate to human (low confidence)")
        history.append((question, "I don't have this information in the documents."))
        return

    docs = [doc for doc, score in results]
    context = "\n\n".join(d.page_content for d in docs)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say exactly: I don't have this information in the documents.

Context:
{context}

Question: {standalone}

Answer:"""

    response = llm.invoke(prompt)
    print("\nAnswer:", response.content)

    print("\nSources:")
    seen = set()
    for d in docs:
        if d.metadata["source"] not in seen:
            seen.add(d.metadata["source"])
            print("-", d.metadata["title"], "|", d.metadata["source"])

    history.append((question, response.content))


In [6]:
history.clear()
ask("How do I change my Notion plan?")
ask("And how do I get a refund?")
ask("What is Notion AI?")
ask("Is it free?")

Best score: 0.65

Answer: To change your Notion plan:

1. In the sidebar, go to **Settings → Billing → Change plan**.  
2. Choose the plan you want to switch to and click **Continue**.  
3. Provide any requested feedback, then click **Downgrade** (or the appropriate confirmation button for the change).  
4. You’ll receive a confirmation of the change and see when your current plan ends.

Sources:
- Change your Notion plan | https://www.notion.com/help/upgrade-or-downgrade-your-plan
- Manage billing in Notion | https://www.notion.com/help/billing
The Understood question: How can I obtain a refund for my Notion plan?
Best score: 0.53

Answer: If you think you signed up for a paid Notion subscription by mistake, you can request a refund by messaging Notion support. They’ll process a full (non‑pro‑rated) refund if you downgrade your subscription within 3 days of signing up for a monthly plan or within 30 days of signing up for an annual plan.

Sources:
- Change your Notion plan | https://w